# Viscan OPG - RetinaNet ResNet-50 FPN V2 (architecture-comparison run)

This notebook trains and evaluates a **second detector architecture** on the exact same data split,
seed, schedule and input scale as the deployed Faster R-CNN ResNet-50 FPN V2 baseline, so the
comparison isolates the architecture. It is self-contained: everything is cloned from a pinned
commit and the dataset is downloaded from the pinned DENTEX revision.

**What gets produced**
- `checkpoints/retinanet/best_model.pth` (best validation mAP@50) and `last_model.pth`
- `runs/retinanet/loss_curve.png`, `runs/retinanet/val_ap_curve.png`
- `evaluation_retinanet/final_evaluation_report.json` (COCO mAP overall + per class, threshold sweep,
  per-class P/R/F1, 5x5 confusion matrix, CPU latency, failure cases)
- `viscan_retinanet_results.zip` with all of the above for download

**Protocol (identical to the baseline run)**
- Fine-tune from COCO_V1 pretrained weights, seed 42, 12 epochs, SGD lr 0.005, cosine schedule, AMP,
  batch size 2, input 800x1333, flip + mild colour jitter.
- The 70-image validation split selects the operating confidence threshold (macro-F1 sweep over
  0.10-0.90). The 50-image test split is evaluated **once**, after training, with that threshold.
- Metrics: COCO mAP@50 / mAP@50:95 / mAP@75 overall and per class; per-class precision/recall/F1 and
  macro-F1; 5x5 confusion matrix with background (FN/FP) accounting; failure-case extraction.

**Before running**
1. Settings -> Accelerator: **GPU T4 x2** (a single T4 is enough; the code uses `cuda:0`).
2. Settings -> Internet: **On** (needed for the GitHub clone and the DENTEX download).
3. Run all. Training 12 epochs on 635 images takes multiple hours; Kaggle GPU sessions are capped at
   12 h. If the session dies mid-training the run must restart from scratch - there is no resume logic.

**Honesty note:** no metric in this notebook is pre-filled. Every number printed comes from the files
this run writes. The only pre-existing values shown are the Faster R-CNN baseline figures, which are
labelled as references copied from `REPORT.md` and are not recomputed here.

**Licences:** torchvision code and both architectures are BSD-3-Clause; DENTEX annotations/images are
CC BY-NC-SA 4.0. The resulting weights inherit the DENTEX non-commercial restriction.


In [ ]:
import os, sys, json, time, platform, subprocess
from pathlib import Path

import torch, torchvision

print("python    :", sys.version.split()[0])
print("torch     :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("platform  :", platform.platform())

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4 x2, then re-run."
print("gpu       :", torch.cuda.get_device_name(0))
print("vram (GB) :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

from torchvision.models.detection import retinanet_resnet50_fpn_v2  # noqa: F401
print("retinanet builder: available")


In [ ]:
REPO_URL = "https://github.com/surendra902/viscan-opg.git"
PINNED_COMMIT = "a5f5d2289e5b2d04a01e41b90fdb3267f4ad5457"
WORKDIR = Path("/kaggle/working/viscan-opg")

if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "--quiet", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(WORKDIR)], check=True)

subprocess.run(["git", "-C", str(WORKDIR), "checkout", "--quiet", PINNED_COMMIT], check=True)
head = subprocess.run(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"],
                      capture_output=True, text=True, check=True).stdout.strip()
assert head == PINNED_COMMIT, f"checkout mismatch: {head}"
print("repo pinned at", head)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

for mod in ("yaml", "pycocotools", "matplotlib", "pandas", "PIL"):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod], check=True)

import model as viscan_model
assert viscan_model.HAS_TORCHVISION, "torchvision detection imports failed"
print("local imports OK | classes:", viscan_model.CLASS_NAMES)


In [ ]:
os.chdir(WORKDIR)

data_dir = Path("data/dentex")
test_images = data_dir / "images" / "test"
ready = ((data_dir / "annotations" / "instances_test.json").exists()
         and test_images.is_dir() and any(test_images.iterdir()))

if ready:
    print("[SKIP] dataset already prepared at", data_dir.resolve())
else:
    print("[RUN] downloading + converting DENTEX (pinned revision) into", data_dir.resolve())
    log = subprocess.run(["bash", "-lc",
        f'set -o pipefail; "{sys.executable}" prepare_data.py data/dentex 2>&1 | tee prepare_data.log'],
        check=False)
    assert log.returncode == 0, "prepare_data.py failed - check prepare_data.log"

import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"free space under /kaggle/working: {free_gb:.1f} GB")


In [ ]:
os.chdir(WORKDIR)
import pandas as pd

data_dir = Path("data/dentex")
splits = ("train", "val", "test")
summary = {}

for s in splits:
    with open(data_dir / "annotations" / f"instances_{s}.json", encoding="utf-8") as f:
        coco = json.load(f)
    per_class = {cid: 0 for cid in viscan_model.CLASS_NAMES if cid != 0}
    for ann in coco["annotations"]:
        per_class[ann["category_id"]] += 1
    summary[s] = {
        "images": len(coco["images"]),
        "boxes": len(coco["annotations"]),
        "per_class": per_class,
        "files": {img["file_name"] for img in coco["images"]},
    }

rows = []
for s in splits:
    row = {"split": s, "images": summary[s]["images"], "boxes": summary[s]["boxes"]}
    for cid, name in viscan_model.CLASS_NAMES.items():
        if cid != 0:
            row[name] = summary[s]["per_class"][cid]
    rows.append(row)
display(pd.DataFrame(rows).set_index("split"))

# Protocol checks: 705-image pool -> 635 train / 70 val (seed 42); official 50-image val used as test.
assert summary["train"]["images"] == 635 and summary["val"]["images"] == 70, "train/val split drift"
assert summary["test"]["images"] == 50, "held-out test split should be the official 50-image val set"
assert summary["train"]["boxes"] + summary["val"]["boxes"] == 3529, "pooled box count should be 3,529"
assert summary["test"]["boxes"] == 182, "held-out test box count should be 182"
assert not (summary["train"]["files"] & summary["val"]["files"]), "train/val filename overlap"
assert not (summary["train"]["files"] & summary["test"]["files"]), "train/test filename overlap"
assert not (summary["val"]["files"] & summary["test"]["files"]), "val/test filename overlap"

print("split verified: 635/70/50 images, 3,529 pooled train+val boxes, 182 held-out test boxes, no filename overlap")


In [ ]:
os.chdir(WORKDIR)
import yaml
from model import build_model_from_config

with open("configs/retinanet.yaml", encoding="utf-8") as f:
    retinanet_cfg = yaml.safe_load(f)

arch = build_model_from_config(retinanet_cfg, pretrained=False)
arch.eval()

cls_head = arch.head.classification_head
anchors = arch.anchor_generator.num_anchors_per_location()[0]
per_class_logits = cls_head.cls_logits.out_channels // anchors
assert per_class_logits == 5, f"expected 5 classes, head has {per_class_logits}"

retinanet_params = sum(p.numel() for p in arch.parameters())
print("architecture      :", retinanet_cfg["model"]["name"])
print("parameters        :", f"{retinanet_params:,}")
print("anchors per loc   :", anchors)
print("logits per class  :", per_class_logits)
print("input scale       :", retinanet_cfg["model"]["min_size"], "x", retinanet_cfg["model"]["max_size"])
print("score/NMS/max dets:", retinanet_cfg["model"]["box_score_thresh"],
      retinanet_cfg["model"]["box_nms_thresh"], retinanet_cfg["model"]["box_detections_per_img"])

del arch
torch.cuda.empty_cache()


In [ ]:
os.chdir(WORKDIR)

train_start = time.time()
proc = subprocess.run(["bash", "-lc",
    f'set -o pipefail; "{sys.executable}" train.py --config configs/retinanet.yaml 2>&1 | tee train_retinanet.log'],
    check=False)
train_minutes = (time.time() - train_start) / 60.0

assert proc.returncode == 0, "training failed - inspect train_retinanet.log"
print(f"\ntraining wall time: {train_minutes:.1f} min")


In [ ]:
os.chdir(WORKDIR)

ckpt_path = Path("checkpoints/retinanet/best_model.pth")
last_path = Path("checkpoints/retinanet/last_model.pth")
assert ckpt_path.exists() and last_path.exists(), "checkpoints missing"

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print("best checkpoint epoch :", ckpt["epoch"])
print("best val mAP@50       :", round(float(ckpt["val_map50"]), 4))
print(f"best checkpoint size  : {ckpt_path.stat().st_size / 1024**2:.0f} MB")

for png in ("runs/retinanet/loss_curve.png", "runs/retinanet/val_ap_curve.png"):
    print(("OK   " if Path(png).exists() else "MISSING ") + png)

del ckpt
torch.cuda.empty_cache()


In [ ]:
os.chdir(WORKDIR)

eval_start = time.time()
proc = subprocess.run(["bash", "-lc",
    f'set -o pipefail; "{sys.executable}" evaluate.py '
    f'--weights checkpoints/retinanet/best_model.pth '
    f'--config configs/retinanet.yaml '
    f'--data_dir data/dentex '
    f'--output_dir evaluation_retinanet '
    f'--skip_variant --select_threshold val 2>&1 | tee eval_retinanet.log'],
    check=False)

assert proc.returncode == 0, "evaluation failed - inspect eval_retinanet.log"
print(f"\nevaluation wall time: {(time.time() - eval_start) / 60.0:.1f} min")


In [ ]:
os.chdir(WORKDIR)
import numpy as np
import pandas as pd

report_path = Path("evaluation_retinanet/final_evaluation_report.json")
assert report_path.exists(), "evaluation report missing"
with open(report_path, encoding="utf-8") as f:
    rep = json.load(f)

print("model :", rep["model"])
print("config:", rep["config"])
print("weights:", rep["weights"])

overall = rep["coco_map"]["overall"]
print("\n--- COCO mAP (held-out 50-image test split) ---")
display(pd.DataFrame([overall]).T.rename(columns={0: "value"}))

print("\n--- Per-class AP ---")
per_class_rows = []
for name, vals in rep["coco_map"]["per_class"].items():
    per_class_rows.append({"class": name, **vals})
display(pd.DataFrame(per_class_rows).set_index("class"))

sel = rep["threshold_selection"]
chosen_tau = sel.get("chosen_threshold", sel.get("best_threshold"))
print("\n--- Threshold selection (validation split only) ---")
print("mode:", sel["mode"], "| chosen threshold:", chosen_tau,
      "| val macro-F1:", sel["best_val_macro_f1"])
display(pd.DataFrame(sel["sweep"]).rename(columns={"threshold": "tau", "macro_f1": "val_macro_f1"}))

tm = rep["threshold_metrics"]
print(f"\n--- Test metrics at tau={tm['confidence_threshold']:.2f} (IoU={tm['iou_threshold']:.2f}) ---")
prf_rows = [{"class": name, **vals} for name, vals in tm["class_metrics"].items()]
display(pd.DataFrame(prf_rows).set_index("class"))
print("test macro-F1:", tm["macro_f1"])

labels = [viscan_model.CLASS_NAMES[cid] for cid in range(5)]
cm = pd.DataFrame(np.array(tm["confusion_matrix"]), index=labels, columns=labels)
cm.index.name = "ground truth \\ predicted"
print("\n--- Confusion matrix (rows = GT, cols = prediction; background = missed / spurious) ---")
display(cm)

print("\n--- CPU latency ---")
print(f"{rep['cpu_latency_ms']:.1f} ms per image (single-image, CPU)")

print("\n--- Failure cases ---")
fail_path = Path("evaluation_retinanet/failures/failure_cases.json")
failures = json.loads(fail_path.read_text(encoding="utf-8")) if fail_path.exists() else []
print("extracted:", rep["num_failure_cases_extracted"], "| listed:", len(failures))
for fc in failures[:5]:
    print(f"- image {fc['image_id']} | {fc['type']} | {fc['finding_class']} | {fc['hypothesis']}")


In [ ]:
os.chdir(WORKDIR)

# Baseline values are COPIED FROM REPORT.md for the existing Faster R-CNN run.
# They are NOT recomputed in this notebook, and the latency was measured on a different machine.
BASELINE_REFERENCE = {
    "source": "REPORT.md - Faster R-CNN ResNet-50 FPN V2, 800x1333, 12 epochs, tau=0.70 selected on val",
    "params": 43_271_528,
    "test_map_50": 0.496,
    "test_map_50_95": 0.329,
    "test_map_75": 0.383,
    "threshold": 0.70,
    "val_macro_f1": 0.556,
    "test_macro_f1": 0.4016,
    "cpu_latency_ms": 4618.0,
}

rn = {
    "params": retinanet_params,
    "test_map_50": overall["map_50"],
    "test_map_50_95": overall["map_50_95"],
    "test_map_75": overall["map_75"],
    "threshold": tm["confidence_threshold"],
    "val_macro_f1": sel["best_val_macro_f1"],
    "test_macro_f1": tm["macro_f1"],
    "cpu_latency_ms": rep["cpu_latency_ms"],
}

def fmt(key, value):
    if key == "params":
        return f"{int(value):,}"
    if key == "threshold":
        return f"{value:.2f}"
    if key == "cpu_latency_ms":
        return f"{value:.0f} ms"
    return f"{value:.4f}"

def delta(key):
    if key == "params":
        return f"{int(rn[key]) - int(BASELINE_REFERENCE[key]):+,}"
    if key == "threshold":
        return f"{rn[key] - BASELINE_REFERENCE[key]:+.2f}"
    if key == "cpu_latency_ms":
        return f"{rn[key] - BASELINE_REFERENCE[key]:+.0f} ms"
    return f"{rn[key] - BASELINE_REFERENCE[key]:+.4f}"

rows = []
for key, label in [
    ("test_map_50", "test mAP@50"),
    ("test_map_50_95", "test mAP@50:95"),
    ("test_map_75", "test mAP@75"),
    ("val_macro_f1", "val macro-F1 (at chosen tau)"),
    ("test_macro_f1", "test macro-F1 (at chosen tau)"),
    ("threshold", "chosen threshold"),
    ("params", "parameters"),
    ("cpu_latency_ms", "CPU latency per image"),
]:
    rows.append({
        "metric": label,
        "Faster R-CNN (REPORT.md reference)": fmt(key, BASELINE_REFERENCE[key]),
        "RetinaNet (this run)": fmt(key, rn[key]),
        "delta (RetinaNet - FRCNN)": delta(key),
    })

print("Faster R-CNN column source:", BASELINE_REFERENCE["source"])
print("NOTE: latency is not apples-to-apples - baseline was measured on a different CPU.\n")
display(pd.DataFrame(rows).set_index("metric"))

print("\nMarkdown row for REPORT.md (generated from this run's JSON, no hand-typed metrics):")
print("| Architecture | test mAP@50 | test mAP@50:95 | test mAP@75 | tau | val macro-F1 | test macro-F1 | params | CPU latency |")
print(f"| RetinaNet ResNet-50 FPN V2 | {rn['test_map_50']:.4f} | {rn['test_map_50_95']:.4f} | "
      f"{rn['test_map_75']:.4f} | {rn['threshold']:.2f} | {rn['val_macro_f1']:.4f} | "
      f"{rn['test_macro_f1']:.4f} | {rn['params']:,} | {rn['cpu_latency_ms']:.0f} ms |")


In [ ]:
os.chdir(WORKDIR)
import zipfile

zip_path = Path("viscan_retinanet_results.zip")
members = [
    ("evaluation_retinanet/final_evaluation_report.json", "final_evaluation_report.json"),
    ("evaluation_retinanet/failures/failure_cases.json", "failure_cases.json"),
    ("runs/retinanet/loss_curve.png", "loss_curve.png"),
    ("runs/retinanet/val_ap_curve.png", "val_ap_curve.png"),
    ("checkpoints/retinanet/best_model.pth", "best_model.pth"),
    ("train_retinanet.log", "train_retinanet.log"),
    ("eval_retinanet.log", "eval_retinanet.log"),
    ("prepare_data.log", "prepare_data.log"),
    ("configs/retinanet.yaml", "retinanet.yaml"),
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
    for src, arcname in members:
        p = Path(src)
        if p.exists():
            zf.write(p, arcname)
            print(f"added {src} ({p.stat().st_size / 1024**2:.1f} MB)")
        else:
            print(f"SKIP (missing) {src}")

print(f"\nzip: {zip_path.resolve()} ({zip_path.stat().st_size / 1024**2:.1f} MB)")


## After the run: how this feeds REPORT.md

Do not change the faster R-CNN numbers. Add the RetinaNet results only after reading the tables above,
and copy the values from `evaluation_retinanet/final_evaluation_report.json` (or from the generated
markdown rows), never from memory.

Suggested additions to `REPORT.md`:

1. **Section 2 / model selection:** one sentence naming RetinaNet ResNet-50 FPN V2 (one-stage, focal
   loss, ResNet-50 FPN V2 backbone, same COCO_V1 pretraining, same 12-epoch schedule and 800x1333
   input) as the architecture-comparison experiment.
2. **Architecture comparison table:** Faster R-CNN reference row vs RetinaNet row for test mAP@50,
   mAP@50:95, mAP@75, chosen tau + val macro-F1, test macro-F1, parameters, CPU latency. Note the
   latency caveat and that the threshold was selected on validation only.
3. **Conclusion:** state whichever architecture actually won, with the measured deltas. If RetinaNet
   loses, keep Faster R-CNN deployed and report the experiment as evidence that the two-stage design
   was the right call - an honest negative result is acceptable and stronger than an unsupported claim.
4. **Provenance:** RetinaNet code BSD-3-Clause (torchvision); weights trained on DENTEX (CC BY-NC-SA
   4.0), so the non-commercial restriction carries over.
5. If RetinaNet wins on accuracy, decide separately whether it is deployable: the packaged Space runs
   Faster R-CNN, so switching would require retraining the 960x1600 variant, re-running the full
   evaluation suite, and updating the Space image.

Optional: move the zip contents into `assets/` and the checkpoint to the Hugging Face weights repo
(`surendra08/viscan-opg-weights`) with a SHA-256 recorded, then cite both in the README.


## Optional cleanup

Run this only after the zip exists. It removes the raw dataset so the Kaggle output stays small; the
trained checkpoint inside the zip is kept. Re-running the notebook afterwards will re-download DENTEX.


In [ ]:
os.chdir(WORKDIR)
import shutil

zip_path = Path("viscan_retinanet_results.zip")
assert zip_path.exists(), "build the zip before cleaning up"

for p in ("data/dentex/zips", "data/dentex/extracted_train", "data/dentex/extracted_val"):
    target = Path(p)
    if target.exists():
        shutil.rmtree(target)
        print("removed", target)

report = Path("evaluation_retinanet/final_evaluation_report.json")
if report.exists():
    shutil.copy2(report, Path("/kaggle/working") / "retinanet_final_evaluation_report.json")
    print("copied report to /kaggle/working/retinanet_final_evaluation_report.json")

print("\nfinal /kaggle/working size:",
      round(sum(f.stat().st_size for f in Path("/kaggle/working").rglob("*") if f.is_file()) / 1024**3, 2),
      "GB")
